# Absenteeism — Phase 1: Preprocessing

**Purpose:** transform the raw `Absenteeism_data.csv` (700 rows, 12 cols) into a clean,
model-ready table and export it as `df_preprocessed.csv`.

**Inputs:** `Absenteeism_data.csv`
**Output:** `df_preprocessed.csv` (15 columns)

Beyond the course steps (drop ID, dummy + group reasons, extract date parts, binarize
Education) this notebook adds a short EDA layer to justify the decisions statistically.

## 0. Imports and load

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

pd.options.display.max_columns = None

In [ ]:
raw_csv = pd.read_csv('Absenteeism_data.csv')
# keep the raw copy untouched; work on df
df = raw_csv.copy()
df.head()

## 1. Initial EDA — understand before transforming

In [ ]:
df.info()

In [ ]:
# no missing values, all numeric except Date
df.describe()

In [ ]:
# the target is heavily right-skewed: most absences are short, a few are very long
# this is why we will later use the MEDIAN (not the mean) as the classification cutoff
df['Absenteeism Time in Hours'].describe()

In [ ]:
# formal check: is the target normally distributed?
w, p = stats.shapiro(df['Absenteeism Time in Hours'])
print(f'Shapiro-Wilk: W={w:.4f}, p={p:.2e}')
print('Not normal (p < 0.05)' if p < 0.05 else 'Normal')
# result: strongly non-normal -> median is the robust choice for the cutoff

## 2. Drop the ID column\nID is an identifier, not a predictive feature.

In [ ]:
df = df.drop(['ID'], axis=1)
df.head()

## 3. Reasons for absence -> dummies -> 4 groups

`Reason for Absence` holds ICD category codes 0-28. We one-hot encode it, drop the
first dummy (code 0, our statistical baseline to avoid multicollinearity), then collapse
the 28 columns into 4 meaningful groups using `.max(axis=1)` (1 if any reason in the
group fired).

In [ ]:
reason_columns = pd.get_dummies(df['Reason for Absence'], drop_first=True).astype(int)
reason_columns.head()

In [ ]:
# group by ICD ranges
reason_type_1 = reason_columns.loc[:, 1:14].max(axis=1)   # diseases
reason_type_2 = reason_columns.loc[:, 15:17].max(axis=1)  # pregnancy related
reason_type_3 = reason_columns.loc[:, 18:21].max(axis=1)  # external causes / injuries
reason_type_4 = reason_columns.loc[:, 22:].max(axis=1)    # light / unspecified

In [ ]:
# drop the original column, then concat the 4 groups
df = df.drop(['Reason for Absence'], axis=1)
df = pd.concat([df, reason_type_1, reason_type_2, reason_type_3, reason_type_4], axis=1)
df.columns = ['Date', 'Transportation Expense', 'Distance to Work', 'Age',
              'Daily Work Load Average', 'Body Mass Index', 'Education',
              'Children', 'Pets', 'Absenteeism Time in Hours',
              'Reason_1', 'Reason_2', 'Reason_3', 'Reason_4']
df.head()

In [ ]:
# reorder so the reason groups come first
df = df[['Reason_1', 'Reason_2', 'Reason_3', 'Reason_4', 'Date',
         'Transportation Expense', 'Distance to Work', 'Age',
         'Daily Work Load Average', 'Body Mass Index', 'Education',
         'Children', 'Pets', 'Absenteeism Time in Hours']]

### Checkpoint 1

In [ ]:
df_checkpoint_reasons = df.copy()
# verify it is a real checkpoint (different object), not a view
print('Independent copy:', df_checkpoint_reasons is not df)

## 4. Date -> month and weekday

Parse the `DD/MM/YYYY` string into a datetime, extract month (1-12) and weekday
(0=Mon ... 6=Sun), then drop the original Date column.

In [ ]:
df['Date'] = pd.to_datetime(df['Date'], format='%d/%m/%Y')
df['Month Value'] = df['Date'].apply(lambda x: x.month)
df['Day of the Week'] = df['Date'].apply(lambda x: x.weekday())
df = df.drop(['Date'], axis=1)

In [ ]:
# reorder to put the new date columns right after the reasons
df = df[['Reason_1', 'Reason_2', 'Reason_3', 'Reason_4', 'Month Value',
         'Day of the Week', 'Transportation Expense', 'Distance to Work', 'Age',
         'Daily Work Load Average', 'Body Mass Index', 'Education',
         'Children', 'Pets', 'Absenteeism Time in Hours']]
df.head()

## 5. Education -> binary

Original: 1=high school, 2=graduate, 3=postgrad, 4=master/doctor. The vast majority are
high school, so we collapse to: 0 = high school, 1 = any higher education.

In [ ]:
# check the distribution first — confirms the imbalance that motivates the binary split
df['Education'].value_counts()

In [ ]:
df['Education'] = df['Education'].map({1: 0, 2: 1, 3: 1, 4: 1})
df['Education'].value_counts()

## 6. Validate against the course reference

In [ ]:
reference = pd.read_csv('df_preprocessed.csv')
print('Same shape:', df.shape == reference.shape)
print('Same content:', df.equals(reference[df.columns]))

## 7. Export

In [ ]:
df_preprocessed = df.copy()
df_preprocessed.to_csv('df_preprocessed_mine.csv', index=False)
print('Saved. Final shape:', df_preprocessed.shape)

## Summary

- Reduced 12 raw columns to 15 model-ready features (4 reason groups + date parts +
  binarized Education + the numeric features + target).
- Confirmed the target is strongly non-normal, justifying the median cutoff used in Phase 2.
- Output matches the course reference exactly.

**Next:** Phase 2 — build the logistic regression in a fresh notebook.